# Maia Lite 0.3.0 — A100 80 GB — benchmark e checkpoint recuperável

**Versão:** 2026-11-04. **Modelo:** 507.026.432 parâmetros, 36 camadas, largura 1024, 16 cabeças, contexto 2048. **Objetivo:** comparar microbatch 2, 4 e 8 em 20 atualizações sintéticas cada; medir tempo, tokens/s e pico de VRAM; testar salvamento/restauração de estados completos.

**Limite explícito:** não consome Maia Corpus v8 e não habilita treinamento com fontes `pending_review`. A aprovação por fonte e o manifesto final de dados continuam pendentes. Este benchmark mede desempenho de GPU, não qualidade do corpus ou do modelo.

**Atenção:** uma rodada de 60 passos completos de um modelo de 507M com contexto 2048 pode demorar e consumir bastante memória. Comece com `STEPS_PER_BATCH = 3` se quiser uma sondagem rápida; depois execute 20. Caso um microbatch produza OOM, ele será registrado como indisponível e os demais serão testados. Reiniciar runtime antes de repetir pode ajudar com fragmentação.

In [ ]:
import os, gc, json, time, random, hashlib, shutil, math
from pathlib import Path
from dataclasses import dataclass, asdict
import torch
from torch import nn
import torch.nn.functional as F
assert torch.cuda.is_available(), 'Selecione GPU A100 no Colab'
assert torch.cuda.is_bf16_supported(), 'GPU sem BF16'
print('GPU:',torch.cuda.get_device_name(0),'PyTorch:',torch.__version__)
print('VRAM total GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
SEED=20261104
random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
model=MaiaGPT(cfg)
params=sum(p.numel() for p in model.parameters())
print('Parameters:',format(params,','),'(~%.3f B)'%(params/1e9))
assert 480_000_000<=params<=530_000_000
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()

## Benchmark sintético de 2048 tokens

**Métrica:** tokens/s = `microbatch × 2048 × passos / tempo` para um passo de otimizador por iteração, sem acumulação. O tempo inclui geração de tokens sintéticos, forward, backward, clipping e AdamW; não inclui carregamento do corpus real. Uma atualização de aquecimento por configuração fica fora das métricas. O primeiro benchmark que inicializa AdamW pode demandar memória adicional. Os testes reutilizam os mesmos pesos em memória e **alteram o modelo**; não use os pesos resultantes para produção.

In [ ]:
STEPS_PER_BATCH=20
WARMUP_STEPS=1
BATCHES=[2,4,8]
SEQ_LEN=2048
LR=1e-4
CLIP_NORM=1.0
OUTPUT_DIR=Path('/content/maia_05b_benchmark')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
# Iniciar com um modelo novo (não reutilizar o modelo do smoke test anterior)
model=model.to('cuda').train()
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(0.9,0.95),weight_decay=0.1,fused=True)
print('Benchmark:',STEPS_PER_BATCH,'passos medidos por microbatch; seq_len:',SEQ_LEN)

In [ ]:
def one_step(batch):
    optimizer.zero_grad(set_to_none=True)
    tokens=torch.randint(0,cfg.vocab_size,(batch,SEQ_LEN+1),device='cuda',dtype=torch.long)
    try:
        with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
            _,loss=model(tokens[:,:-1],tokens[:,1:])
        if not torch.isfinite(loss):
            raise FloatingPointError('Loss não finita')
        loss.backward()
        grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP_NORM,error_if_nonfinite=True)
        optimizer.step()
        return float(loss.detach()),float(grad_norm.detach())
    finally:
        del tokens

results=[]
for batch in BATCHES:
    print(f'\n--- microbatch={batch} ---',flush=True)
    gc.collect();torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        for _ in range(WARMUP_STEPS):one_step(batch)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        t0=time.perf_counter();losses=[];norms=[]
        for i in range(STEPS_PER_BATCH):
            loss,gn=one_step(batch)
            losses.append(loss);norms.append(gn)
            if (i+1)%5==0 or i+1==STEPS_PER_BATCH:
                print(f'  step {i+1}/{STEPS_PER_BATCH} loss={loss:.4f}',flush=True)
        torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        result={'microbatch':batch,'status':'OK','steps':STEPS_PER_BATCH,
                'seq_len':SEQ_LEN,'elapsed_s':round(elapsed,3),
                'seconds_per_step':round(elapsed/STEPS_PER_BATCH,3),
                'tokens_per_second':round(batch*SEQ_LEN*STEPS_PER_BATCH/elapsed,1),
                'peak_allocated_gib':round(torch.cuda.max_memory_allocated()/2**30,2),
                'peak_reserved_gib':round(torch.cuda.max_memory_reserved()/2**30,2),
                'last_loss':round(losses[-1],5),'last_grad_norm':round(norms[-1],4)}
    except (torch.cuda.OutOfMemoryError,RuntimeError) as exc:
        if not isinstance(exc,torch.cuda.OutOfMemoryError) and 'out of memory' not in str(exc).lower():
            raise
        optimizer.zero_grad(set_to_none=True)
        gc.collect();torch.cuda.empty_cache()
        result={'microbatch':batch,'status':'OOM','error':str(exc)[:400]}
    results.append(result)
    print(json.dumps(result,ensure_ascii=False),flush=True)

report={'model_parameters':sum(p.numel() for p in model.parameters()),
        'gpu':torch.cuda.get_device_name(0),'synthetic_only':True,
        'rights_approval':False,'results':results}
report_path=OUTPUT_DIR/'benchmark_results.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('\nRelatório:',report_path)
valid=[r for r in results if r['status']=='OK']
if valid:
    best=max(valid,key=lambda r:r['tokens_per_second'])
    print('MELHOR THROUGHPUT:',best['microbatch'],'microbatch;',best['tokens_per_second'],'tokens/s')
else:
    print('Nenhum microbatch completou. Tente reiniciar runtime e testar batch=1.')

## Configuração proposta após o benchmark

A melhor vazão sintética é apenas um indicador. Antes do treino real, repetir com o carregador aprovado, validar throughput de I/O e avaliar se há espaço para acumulação de gradientes. Com `microbatch=4`, `grad_accum=8`, o batch efetivo seria 32 sequências = 65.536 tokens por atualização. **Não extrapolar os 2 segundos do smoke test de contexto 256 para contexto 2048.**

In [ ]:
valid=[r for r in results if r['status']=='OK']
if valid:
    best=max(valid,key=lambda r:r['tokens_per_second'])
    MICRO_BATCH=best['microbatch']
    TARGET_TOKENS_PER_UPDATE=65536
    GRAD_ACCUM=max(1,math.ceil(TARGET_TOKENS_PER_UPDATE/(MICRO_BATCH*SEQ_LEN)))
    print('Microbatch sugerido:',MICRO_BATCH)
    print('Gradient accumulation sugerido:',GRAD_ACCUM)
    print('Tokens por atualização:',MICRO_BATCH*SEQ_LEN*GRAD_ACCUM)
    print('Estimativa tokens/s sintéticos:',best['tokens_per_second'])
else:
    print('Benchmark não produziu configuração válida.')

## Checkpoint completo — teste de retomada (somente dados sintéticos)

Checkpoint inclui pesos, AdamW, RNG CPU/CUDA/Python, arquitetura, estado de scheduler e posição sintética. **Não é checkpoint de produção**. Salvamento em `/content` é temporário; para persistência, configure explicitamente um caminho no Drive com espaço livre suficiente. O checkpoint completo pode ultrapassar **6 GiB**. Salvamos primeiro em `.tmp`, sincronizamos e renomeamos; para Google Drive, essa operação reduz risco de arquivo incompleto, mas não substitui verificação externa de integridade. O teste abaixo é opt-in para evitar escrita desnecessária de gigabytes.

In [ ]:
SAVE_FULL_CHECKPOINT=False  # mude para True para testar gravação/restauração
CHECKPOINT_DIR=Path('/content/maia_05b_benchmark/checkpoints')
# Para salvar no Drive, monte-o e altere CHECKPOINT_DIR para uma pasta sua.
if SAVE_FULL_CHECKPOINT:
    CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)
    checkpoint_path=CHECKPOINT_DIR/'synthetic_full_step.pt'
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lambda step:1.0)
    # O benchmark não usa scheduler. Estado inicial salvo para demonstrar serialização.
    state={'schema_version':1,'source':'SYNTHETIC_ONLY','training_approved':False,
           'model_config':asdict(cfg),'model':model.state_dict(),
           'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
           'global_step':sum(r.get('steps',0)+WARMUP_STEPS for r in results if r['status']=='OK'),
           'data_cursor':{'type':'synthetic','position':None},
           'python_rng':random.getstate(),'torch_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all()}
    tmp=checkpoint_path.with_suffix('.pt.tmp')
    torch.save(state,tmp)
    with open(tmp,'rb') as f:os.fsync(f.fileno())
    os.replace(tmp,checkpoint_path)
    print('Checkpoint salvo:',checkpoint_path,'GiB:',round(checkpoint_path.stat().st_size/2**30,2))
    # weights_only=False apenas porque este checkpoint foi produzido localmente por esta célula.
    restored=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
    assert restored['source']=='SYNTHETIC_ONLY'
    assert restored['model_config']==asdict(cfg)
    assert restored['global_step']==state['global_step']
    assert len(restored['optimizer']['state'])>0
    print('Checkpoint lido e campos essenciais verificados.')
    del restored,state
else:
    print('Checkpoint completo não gravado. Altere SAVE_FULL_CHECKPOINT=True se desejar testar.')

## Portão de conformidade e próximos requisitos

- O snapshot `wikimedia/wikipedia` revisão `b04c8d1ceb2f5cd4588862100d08de323dccfbaa`, configuração `20231101.pt`, foi identificado pelo usuário; os 650 chunks auditados permanecem `pending_review`, `eligible_for_training=false`.
- O manifesto final de produção deve identificar, **por fonte e por shard**, direitos, SHA-256, split, tokenizer, orçamento de tokens, IDs especiais, exclusão de duplicatas e a decisão documentada de elegibilidade.
- O pré-treinamento real deve salvar checkpoint completo e cursor do sampler em conjunto, com retomada determinística; o checkpoint de teste acima **não demonstra** isso com dados reais.
- Não basta a existência de um `license_approval.json`: seu conteúdo, escopo, assinaturas e hashes devem ser validados antes de habilitar dados reais.
- O código não lê os shards reais, não altera o SQLite v5 e não muda direitos de qualquer registro.

**Envie a tabela JSON de benchmark**, especialmente `tokens_per_second` e `peak_allocated_gib`, para fixarmos a configuração de produção.